# Notebook 02b — LIME Explanations

This notebook generates LIME explanations for the same 500 instances sampled in 02a.

**Run 02a first** — this notebook depends on `sample_indices.npy`.

**Inputs** (from Drive):
- `adult_test.csv` — encoded test set with sensitive columns attached
- `adult_train.csv` — training set (used to fit the LIME explainer background)
- `model_rf.pkl` — trained Random Forest classifier
- `feature_names.pkl` — list of 14 feature names
- `sample_indices.npy` — produced by notebook 02a
- `shap_values.npy` — produced by notebook 02a (for comparison plot)

**Outputs** (saved to Drive):
- `lime_values.npy` — LIME weights, shape (500, 14)

**Runtime note**: LIME trains a local linear model per instance.
Expect ~15-30 minutes for 500 instances on Colab.

## Step 1 — Install and Import Libraries

In [ ]:
!pip install lime -q

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from google.colab import drive

print(f"lime version: {lime.__version__}")

## Step 2 — Mount Drive and Load Artifacts

In [ ]:
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/XAIP/data/'
print(f"✓ Drive mounted — base path: {BASE}")

In [ ]:
# Load test set
df_test = pd.read_csv(BASE + 'adult_test.csv')

# Load training set (needed for LIME background distribution)
df_train = pd.read_csv(BASE + 'adult_train.csv')

# Load model and feature names
with open(BASE + 'model_rf.pkl', 'rb') as f:
    model = pickle.load(f)
with open(BASE + 'feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

# Load sample indices from 02a
sample_idx = np.load(BASE + 'sample_indices.npy')

# Separate features for test set
X_test  = df_test[feature_names].copy()
y_test  = df_test['income'].copy()

# Training features for LIME background
X_train = df_train[feature_names].copy()

print(f"Test set:          {X_test.shape}")
print(f"Train set:         {X_train.shape}")
print(f"Sample indices:    {len(sample_idx)} instances")
print(f"Features ({len(feature_names)}):  {feature_names}")

## Step 3 — Reconstruct Sample

We reload the same 500 instances from 02a using the saved indices.
This guarantees SHAP and LIME explanations are on identical instances.

In [ ]:
X_sample = X_test.loc[sample_idx].reset_index(drop=True)
y_sample = y_test.loc[sample_idx].reset_index(drop=True)

N_SAMPLE = len(X_sample)
assert N_SAMPLE == 500, f"Expected 500 samples, got {N_SAMPLE}"

print(f"X_sample shape: {X_sample.shape}")
print(f"y_sample dist:\n{y_sample.value_counts()}")

## Step 4 — Initialize LIME Explainer

We use the full training set as the background distribution.

Key parameters:
- `mode='classification'` — binary output
- `discretize_continuous=True` — LIME bins continuous features; we still extract
  weights by feature index (not by bin label) so the output matrix is clean
- `random_state=42` — reproducibility

In [ ]:
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=feature_names,
    class_names=['<=50K', '>50K'],
    mode='classification',
    discretize_continuous=True,
    random_state=42
)

print("✓ LimeTabularExplainer initialized")
print(f"  Background set size: {len(X_train)}")
print(f"  Number of features:  {len(feature_names)}")

## Step 5 — Generate LIME Explanations

For each of the 500 instances we:
1. Call `explain_instance` with `num_features=len(feature_names)` to get weights
   for all 14 features.
2. Extract `local_exp[1]` — a list of `(feature_index, weight)` pairs for class 1.
3. Fill a dense row in `lime_values` using the feature indices directly,
   avoiding the need to parse LIME's discretized label strings.

This loop is the slow part — expect 15-30 minutes.

In [ ]:
NUM_SAMPLES_LIME = 1000  # perturbations LIME generates per instance

lime_values = np.zeros((N_SAMPLE, len(feature_names)), dtype=np.float64)

for i in tqdm(range(N_SAMPLE), desc='LIME explanations'):
    instance = X_sample.iloc[i].values

    exp = lime_explainer.explain_instance(
        data_row=instance,
        predict_fn=model.predict_proba,
        num_features=len(feature_names),
        num_samples=NUM_SAMPLES_LIME
    )

    # local_exp[1] = [(feature_index, weight), ...] for the positive class
    for feat_idx, weight in exp.local_exp[1]:
        lime_values[i, feat_idx] = weight

print(f"\n✓ LIME complete")
print(f"  lime_values shape: {lime_values.shape}")
print(f"  Non-zero entries:  {np.count_nonzero(lime_values)} / {lime_values.size}")

## Step 6 — Save LIME Values

In [ ]:
np.save(BASE + 'lime_values.npy', lime_values)
print(f"✓ lime_values.npy saved — shape {lime_values.shape}, dtype {lime_values.dtype}")

# Reload verification
lime_reload = np.load(BASE + 'lime_values.npy')
assert np.allclose(lime_values, lime_reload)
print("✓ Reload verification passed")

## Step 7 — Visualizations

### 7a — Global Feature Importance (Mean |LIME weight|)

In [ ]:
mean_abs_lime = np.abs(lime_values).mean(axis=0)
importance_df = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_lime': mean_abs_lime
}).sort_values('mean_abs_lime', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(importance_df['feature'], importance_df['mean_abs_lime'], color='darkorange')
ax.set_xlabel('Mean |LIME weight|')
ax.set_title('LIME Global Feature Importance (n=500)')
plt.tight_layout()
plt.savefig(BASE + 'lime_importance.png', dpi=150)
plt.show()

print("\nTop 5 features by mean |LIME weight|:")
print(importance_df.sort_values('mean_abs_lime', ascending=False).head(5).to_string(index=False))

### 7b — Example: LIME Explanation for a Single Instance

In [ ]:
# Re-explain one instance to show the LIME visualization
pred_proba = model.predict_proba(X_sample)[:, 1]
example_idx = int(np.argmax(pred_proba))

print(f"Example instance index: {example_idx}")
print(f"  Predicted P(>50K):  {pred_proba[example_idx]:.3f}")
print(f"  True label:         {y_sample.iloc[example_idx]}")

example_exp = lime_explainer.explain_instance(
    data_row=X_sample.iloc[example_idx].values,
    predict_fn=model.predict_proba,
    num_features=14,
    num_samples=NUM_SAMPLES_LIME
)
example_exp.show_in_notebook(show_table=True)

### 7c — SHAP vs LIME: Feature Ranking Comparison

Ranks features by mean absolute weight for each method and plots them side by side.
This is a sanity check and a useful figure for the paper.

In [ ]:
shap_values = np.load(BASE + 'shap_values.npy')

mean_abs_shap = np.abs(shap_values).mean(axis=0)
mean_abs_lime = np.abs(lime_values).mean(axis=0)

# Rank from most to least important (rank 1 = most important)
shap_ranks = pd.Series(mean_abs_shap, index=feature_names).rank(ascending=False).astype(int)
lime_ranks  = pd.Series(mean_abs_lime,  index=feature_names).rank(ascending=False).astype(int)

comparison_df = pd.DataFrame({
    'SHAP rank': shap_ranks,
    'LIME rank':  lime_ranks,
    'rank_diff':  (shap_ranks - lime_ranks).abs()
}).sort_values('SHAP rank')

print("Feature ranking comparison (SHAP vs LIME):")
print(comparison_df.to_string())
print(f"\nMean absolute rank difference: {comparison_df['rank_diff'].mean():.2f}")

# Side-by-side bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

shap_order = pd.Series(mean_abs_shap, index=feature_names).sort_values()
lime_order  = pd.Series(mean_abs_lime,  index=feature_names).sort_values()

axes[0].barh(shap_order.index, shap_order.values, color='steelblue')
axes[0].set_title('SHAP — Mean |value|')
axes[0].set_xlabel('Mean |SHAP value|')

axes[1].barh(lime_order.index, lime_order.values, color='darkorange')
axes[1].set_title('LIME — Mean |weight|')
axes[1].set_xlabel('Mean |LIME weight|')

plt.suptitle('SHAP vs LIME Global Feature Importance (n=500)', fontsize=13)
plt.tight_layout()
plt.savefig(BASE + 'shap_vs_lime_importance.png', dpi=150)
plt.show()

## Step 8 — Summary

In [ ]:
print("=" * 60)
print("  SUMMARY — OUTPUTS FOR NOTEBOOK 03")
print("=" * 60)
print(f"\nBASE PATH:\n  {BASE}")
print(f"\nFILES PRODUCED:")
files = ['lime_values.npy', 'lime_importance.png', 'shap_vs_lime_importance.png']
for fname in files:
    fpath = BASE + fname
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  {fname:35s}  {size_kb:.1f} KB")

print(f"\nLIME VALUES:")
print(f"  Shape:   {lime_values.shape}  (instances × features)")
print(f"  dtype:   {lime_values.dtype}")
print(f"  min:     {lime_values.min():.4f}")
print(f"  max:     {lime_values.max():.4f}")
print(f"  mean|w|: {np.abs(lime_values).mean():.4f}")

print(f"\nALL EXPLAINER ARTIFACTS NOW IN DRIVE:")
all_artifacts = [
    'sample_indices.npy', 'X_sample.csv', 'y_sample.csv', 'sens_sample.csv',
    'shap_values.npy', 'lime_values.npy'
]
for fname in all_artifacts:
    exists = '✓' if os.path.exists(BASE + fname) else '✗ MISSING'
    print(f"  {exists}  {fname}")

print(f"\n✓ LIME complete — ready for notebook 03 (metrics)")